# Symbolic MIDI Generation Draft Workbook

This draft documents the current Assignment 2 pipeline for two symbolic music generation tasks: unconditioned MIDI generation and prefix-conditioned MIDI continuation. It is a working report draft, not the final exported submission.

## 1. Introduction and Task Definitions

The project treats symbolic MIDI generation as next-token language modeling over MIDI-derived event tokens. A shared model can support both required tasks:

- **Task 1: symbolic unconditioned generation.** Sample a new token sequence from a beginning seed and decode it to MIDI.
- **Task 2: symbolic prefix-conditioned continuation.** Encode a real MIDI prefix, use it as the prompt, and sample a continuation.

The main neural model is a GPT-2-style causal Transformer initialized from scratch with a custom MIDI vocabulary. No pretrained GPT-2 weights, pretrained music checkpoints, or GPT-2 text tokenizer are used.

## 2. Dataset and Preprocessing

The current draft includes a Nottingham MIDI subset and a small MAESTRO MIDI-only subset. Audio was not downloaded or used. Files are split into train/validation partitions, tokenized, and converted into fixed-length next-token windows.

### Dataset Summary

| dataset | file_count | train_files | valid_files | token_min | token_max | token_mean | train_windows | valid_windows | vocab_size |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| nottingham | 150 | 120 | 30 | 57 | 3243 | 277.8666666666667 | 761 | 314 | 512 |
| maestro | 40 | 32 | 8 | 3087 | 37808 | 11963.2 | 6033 | 1422 | 512 |

## 3. Tokenization

The primary representation is MidiTok REMI. REMI represents symbolic music with discrete musical events such as bar, position, pitch, velocity, and duration. This keeps the model in a language-modeling setting while still preserving musical timing structure.

A simple custom tokenizer remains the fallback for smoke tests if MidiTok decoding becomes unstable, but the current real-data runs use REMI successfully.

## 4. Markov / N-Gram Baseline

The Markov baseline estimates next-token probabilities from local token histories. It gives a simple, reliable reference point for valid MIDI generation and validation perplexity.

### Model Metrics

| dataset | markov_valid_perplexity | transformer_train_loss_last | transformer_valid_loss | transformer_valid_perplexity | transformer_params | block_size | steps_completed |
| --- | --- | --- | --- | --- | --- | --- | --- |
| nottingham | 72.89547291034337 | 5.149081707000732 | 5.251197934150696 | 190.79469108990625 | 136960 | 64 | 80 |
| maestro | 296.8909029097737 | 5.70381498336792 | 5.679322842801555 | 292.7511242569301 | 136960 | 64 | 80 |

## 5. GPT2-Style Causal Transformer Trained From Scratch

The neural model uses `GPT2Config` and `GPT2LMHeadModel(config)` as a decoder-only Transformer architecture. The model is randomly initialized and trained on MIDI token windows. The current run is bounded and intentionally small, so the results should be interpreted as a credible first draft rather than final-quality music.

## 6. Task 1: Symbolic Unconditioned Generation

For unconditioned generation, the sampler starts from a short seed and generates new MIDI tokens. Candidate files are decoded, parsed, and ranked by validity, note count, duration, pitch range, polyphony, and repetition heuristics.

## 7. Task 2: Prefix-Conditioned Continuation

For conditioned continuation, a validation MIDI prefix is used as the prompt. The model samples additional tokens after that prefix, and the resulting sequence is decoded to MIDI. This tests whether the same next-token model can generate in context.

## 8. Evaluation

Candidate MIDI files are checked for parseability and nonzero notes. The ranking table below is a rough quantitative screen, not a substitute for listening.

### Candidate Ranking

| path | dataset | task_type | model_type | valid | note_count | duration_seconds | notes_per_second | pitch_min | pitch_max | pitch_range | unique_pitch_count | max_simultaneous_notes | repeated_pitch_bigram_rate | score |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\nottingham_subset\markov_conditioned.mid | nottingham | conditioned | markov | True | 130 | 40.5 | 3.2098765432098766 | 60 | 83 | 23 | 20 | 4 | 0.4031007751937984 | 0.9137058570198104 |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\nottingham_subset\markov_unconditioned.mid | nottingham | unconditioned | markov | True | 130 | 50.75 | 2.561576354679803 | 62 | 83 | 21 | 18 | 2 | 0.4728682170542636 | 0.9297493349117248 |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\nottingham_subset\transformer_conditioned.mid | nottingham | conditioned | transformer | True | 74 | 16.0 | 4.625 | 62 | 79 | 17 | 13 | 7 | 0.3835616438356164 | -0.0483732876712328 |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\nottingham_subset\transformer_unconditioned.mid | nottingham | unconditioned | transformer | True | 48 | 24.0 | 2.0 | 62 | 79 | 17 | 7 | 6 | 0.4042553191489361 | 0.2303782505910164 |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\maestro_subset\markov_conditioned.mid | maestro | conditioned | markov | True | 87 | 11.875 | 7.326315789473684 | 35 | 94 | 59 | 38 | 7 | 0.0813953488372093 | 0.7139563783489733 |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\maestro_subset\markov_unconditioned.mid | maestro | unconditioned | markov | True | 92 | 6.875 | 13.38181818181818 | 29 | 89 | 60 | 42 | 9 | 0.0219780219780219 | -0.3498398823398822 |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\maestro_subset\transformer_conditioned.mid | maestro | conditioned | transformer | True | 10 | 2.75 | 3.636363636363636 | 42 | 70 | 28 | 6 | 3 | 0.2222222222222222 | 0.1809343434343435 |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\maestro_subset\transformer_unconditioned.mid | maestro | unconditioned | transformer | True | 92 | 6.875 | 13.38181818181818 | 29 | 89 | 60 | 42 | 9 | 0.0219780219780219 | -0.3498398823398822 |

### Selected Current Candidates

- nottingham unconditioned: `outputs\candidates\selected\nottingham\unconditioned_markov.mid` (source `outputs\candidates\nottingham_subset\markov_unconditioned.mid`)
- nottingham conditioned: `outputs\candidates\selected\nottingham\conditioned_markov.mid` (source `outputs\candidates\nottingham_subset\markov_conditioned.mid`)
- maestro unconditioned: `outputs\candidates\selected\maestro\unconditioned_transformer.mid` (source `outputs\candidates\maestro_subset\transformer_unconditioned.mid`)
- maestro conditioned: `outputs\candidates\selected\maestro\conditioned_markov.mid` (source `outputs\candidates\maestro_subset\markov_conditioned.mid`)

### Token Length Distributions

![Nottingham token length distribution](../outputs/evaluation/figures/nottingham_token_lengths.png)

_Nottingham token length distribution_

![MAESTRO token length distribution](../outputs/evaluation/figures/maestro_token_lengths.png)

_MAESTRO token length distribution_

### Pitch-Class Histograms

![Nottingham train vs selected generated pitch-class histogram](../outputs/evaluation/figures/nottingham_pitch_class_histogram.png)

_Nottingham train vs selected generated pitch-class histogram_

![MAESTRO train vs selected generated pitch-class histogram](../outputs/evaluation/figures/maestro_pitch_class_histogram.png)

_MAESTRO train vs selected generated pitch-class histogram_

## 9. Related Work Notes

This project is aligned with symbolic music generation methods from the course material, especially next-event prediction over symbolic music representations. The most relevant references for the final writeup are REMI / Pop Music Transformer, Music Transformer, Performance RNN-style symbolic sequence modeling, Markov and n-gram baselines, Nottingham, and MAESTRO.

## 10. Discussion, Limitations, and Future Work

The pipeline now produces valid MIDI candidates for both tasks. The main limitations are musical quality, short bounded training, and heuristic candidate selection. The next pass should listen to the selected files, choose the final dataset route, train the selected Transformer longer if time allows, and add qualitative observations.

## 11. Current Artifacts and Next Steps

Current generated artifacts live under `outputs/`, including metrics tables, figures, and selected candidate MIDI files. These are draft artifacts only. Final submission files have not been created yet.

Before submission, export this workbook to HTML, copy the selected MIDI files into `submission/` with the required names, and add the video URL file after recording the presentation.